## **01_Data_Collection**

This notebook handles the **raw data collection pipeline** for Google Play Store reviews across four Indonesian ride-hailing platforms: Grab, Gojek, Maxim, and InDrive. It uses the `google-play-scraper` Python library to programmatically pull up to 25,000 reviews per platform (100,000 total) directly from the Google Play Store, filtering for Indonesian-language reviews (`lang='id'`, `country='id'`) sorted by newest first. Reviews are scraped in batches of 1,000 using a pagination token (`continuation_token`) to navigate across pages, with a 3-second delay between batches and a 5-second delay between platforms to avoid IP blocking. Each platform's data is saved as a separate CSV file to `data/raw/google_play_reviews/`. A second cell — currently commented out — contains a parallel scraping script for the Apple App Store using `app-store-scraper`, with schema alignment to match the Google Play format. This cell was not executed in the final pipeline, so the project relies solely on Google Play data.

# **Scraping from Google Play**

This cell contains the complete Google Play Store data collection pipeline. It begins by importing `pandas`, `google_play_scraper`, `os`, and `time`, then creates the output directory `data/raw/google_play_reviews/` if it does not already exist.

A dictionary `ride_hailing_apps` maps each platform name to its Google Play package ID: `com.grabtaxi.passenger` (Grab), `com.gojek.app` (Gojek), `com.taxsee.taxsee` (Maxim), and `sinet.startup.inDriver` (InDrive). Two constants are defined: `TARGET_REVIEWS = 25,000` (maximum reviews to collect per app) and `BATCH_SIZE = 1,000` (reviews pulled per API request).

The core `scrape_play_store_reviews()` function implements a `while` loop that repeatedly calls `reviews()` from `google_play_scraper`, passing a `continuation_token` to paginate through results. On each iteration it fetches up to `BATCH_SIZE` reviews, appends them to `all_results`, and sleeps for 3 seconds before the next batch. The loop terminates when either the target count is reached or the API returns `None` for the continuation token (no more reviews available). The raw results are converted to a DataFrame and filtered to 9 columns: `reviewId`, `userName`, `content`, `score`, `thumbsUpCount`, `reviewCreatedVersion`, `at`, `replyContent`, and `repliedAt`. A 5-row preview is printed before saving each platform's data as a UTF-8 CSV.

In [32]:
import pandas as pd
from google_play_scraper import Sort, reviews
import os
import time

# 1. Menyiapkan direktori output
output_dir = '../data/raw/google_play_reviews'
os.makedirs(output_dir, exist_ok=True)

# 2. Dictionary berisi nama aplikasi
ride_hailing_apps = {
    'grab': 'com.grabtaxi.passenger',
    'gojek': 'com.gojek.app',
    'maxim': 'com.taxsee.taxsee',
    'indrive': 'sinet.startup.inDriver'
}

# Target jumlah review dinaikkan menjadi 25.000 / app
TARGET_REVIEWS = 25000 
# Batas penarikan per satu kali request agar aman dari blokir
BATCH_SIZE = 1000 

def scrape_play_store_reviews(app_name, app_id, target_count):
    print(f"⏳ Memulai scraping target {target_count} ulasan untuk {app_name.capitalize()}...")
    
    all_results = []
    continuation_token = None
    fetched_count = 0
    
    try:
        # Loop akan terus berjalan sampai target 50.000 terpenuhi
        while fetched_count < target_count:
            # Mengatur agar tarikan terakhir tidak melebihi batas target
            count_to_fetch = min(BATCH_SIZE, target_count - fetched_count)
            
            result, continuation_token = reviews(
                app_id,
                lang='id',           
                country='id',        
                sort=Sort.NEWEST,    
                count=count_to_fetch,         
                continuation_token=continuation_token # Kunci untuk lanjut ke halaman berikutnya
            )
            
            # Gabungkan hasil tarikan ke list utama
            all_results.extend(result)
            fetched_count += len(result)
            print(f"   ➔ Terambil: {fetched_count} / {target_count} ulasan...")
            
            # Berhenti jika server Google menyatakan ulasan sudah habis
            if continuation_token is None:
                print("   ➔ Tidak ada lagi ulasan yang tersedia di server.")
                break
                
            # Jeda krusial agar IP tidak diblokir oleh Google
            time.sleep(3)
        
        # Konversi ke DataFrame
        df = pd.DataFrame(all_results)
        
        # Filter kolom
        columns_to_keep = [
            'reviewId', 'userName', 'content', 'score', 
            'thumbsUpCount', 'reviewCreatedVersion', 'at', 
            'replyContent', 'repliedAt'
        ]
        
        if not df.empty:
            df = df[columns_to_keep]
            
            # --- BAGIAN UNTUK MENAMPILKAN PREVIEW DATA ---
            print(f"\n👀 Preview 5 data teratas untuk {app_name.capitalize()}:")
            # df.head() akan mencetak 5 baris pertama secara otomatis
            print(df.head()) 
            print("-" * 50)
            
            # 3. Menyimpan DataFrame ke dalam format CSV
            file_path = os.path.join(output_dir, f"{app_name}_reviews.csv")
            df.to_csv(file_path, index=False, encoding='utf-8')
            
            print(f"✅ Selesai! Data disimpan di: {file_path} | Total baris akhir: {len(df)}\n")
        else:
             print(f"⚠️ Tidak ada ulasan yang berhasil diekstrak untuk {app_name}.\n")
             
    except Exception as e:
        print(f"❌ Terjadi kesalahan saat scraping {app_name}: {e}\n")

# 4. Eksekusi loop untuk ke-4 platform
print("🚀 Memulai Data Collection Pipeline (Mode Ekstraksi Besar)...\n" + "="*40)

for name, package_id in ride_hailing_apps.items():
    scrape_play_store_reviews(name, package_id, target_count=TARGET_REVIEWS)
    # Jeda yang lebih lama antar aplikasi
    time.sleep(5) 

print("🎉 Seluruh proses scraping telah selesai.")

🚀 Memulai Data Collection Pipeline (Mode Ekstraksi Besar)...
⏳ Memulai scraping target 25000 ulasan untuk Grab...


   ➔ Terambil: 1000 / 25000 ulasan...
   ➔ Terambil: 2000 / 25000 ulasan...
   ➔ Terambil: 3000 / 25000 ulasan...
   ➔ Terambil: 4000 / 25000 ulasan...
   ➔ Terambil: 5000 / 25000 ulasan...
   ➔ Terambil: 6000 / 25000 ulasan...
   ➔ Terambil: 7000 / 25000 ulasan...
   ➔ Terambil: 8000 / 25000 ulasan...
   ➔ Terambil: 9000 / 25000 ulasan...
   ➔ Terambil: 10000 / 25000 ulasan...
   ➔ Terambil: 11000 / 25000 ulasan...
   ➔ Terambil: 12000 / 25000 ulasan...
   ➔ Terambil: 13000 / 25000 ulasan...
   ➔ Terambil: 14000 / 25000 ulasan...
   ➔ Terambil: 15000 / 25000 ulasan...
   ➔ Terambil: 16000 / 25000 ulasan...
   ➔ Terambil: 17000 / 25000 ulasan...
   ➔ Terambil: 18000 / 25000 ulasan...
   ➔ Terambil: 19000 / 25000 ulasan...
   ➔ Terambil: 20000 / 25000 ulasan...
   ➔ Terambil: 21000 / 25000 ulasan...
   ➔ Terambil: 22000 / 25000 ulasan...
   ➔ Terambil: 23000 / 25000 ulasan...
   ➔ Terambil: 24000 / 25000 ulasan...
   ➔ Terambil: 25000 / 25000 ulasan...

👀 Preview 5 data teratas untuk Gr

> All four platforms successfully reached 25,000 reviews each, producing a combined raw dataset of **100,000 rows** across four CSV files. The most recent reviews scraped are dated 27 May 2026.